# Imports and setup

In [5]:
import pandas as pd

from demand.baselines import naive, seasonal_naive, seasonal_average
from demand.data import clean, load_subset
from demand.metrics import evaluate
from demand.validation import split_holdout, walk_forward_splits

# Load and clean data

In [6]:
df = clean(load_subset())
train_part, holdout = split_holdout(df)

print("rows:", len(df), "| days:", df.date.nunique(), "| series:", df.groupby(["store_nbr", "family"], observed=True).ngroups)
print("train part:", train_part.date.min().date(), "->", train_part.date.max().date())
print("hold-out:", holdout.date.min().date(), "->", holdout.date.max().date())
print("closed days:", sorted(df[df.is_closed].date.dt.date.unique()))
print("NaN left:", df.isna().sum().sum())

rows: 18275 | days: 731 | series: 25
train part: 2015-08-16 -> 2017-07-18
hold-out: 2017-07-19 -> 2017-08-15
closed days: [datetime.date(2015, 12, 25), datetime.date(2016, 1, 1), datetime.date(2016, 12, 25), datetime.date(2017, 1, 1)]
NaN left: 0


# Walk-forward evaluation

In [7]:
def run_walk_forward(model):
    rows = []
    for fold, (train_dates, val_dates) in enumerate(walk_forward_splits(train_part.date), 1):
        train = train_part[train_part.date.isin(train_dates)]
        val = train_part[train_part.date.isin(val_dates)]
        pred = model(train, val)
        assert pred.pred.notna().all()
        open_days = ~pred.is_closed
        rows.append({"fold": fold, **evaluate(pred.sales[open_days], pred.pred[open_days])})
    return pd.DataFrame(rows).set_index("fold")


models = {"naive": naive, "seasonal_naive": seasonal_naive, "seasonal_median_4w": seasonal_average}
results = {name: run_walk_forward(model) for name, model in models.items()}

display("seasonal_naive")
display(results["seasonal_naive"].round(3))

display("naive")
display(results["naive"].round(3))

display("seasonal_median_4w")
display(results["seasonal_median_4w"].round(3))

display("General evaluation")
pd.DataFrame({name: r.mean() for name, r in results.items()}).T.round(3)

'seasonal_naive'

,MAE,RMSE,WAPE
fold,,,
1,1226.789,2796.604,0.219
2,2324.435,3914.209,0.473
3,691.925,1100.936,0.146
4,1038.672,1888.230,0.213
5,642.435,1147.328,0.137
6,668.135,1203.910,0.139
7,640.622,1078.403,0.133
8,647.496,1015.071,0.138


'naive'

,MAE,RMSE,WAPE
fold,,,
1,1295.698,1891.651,0.232
2,1495.355,2112.327,0.305
3,991.927,1710.542,0.210
4,1280.523,1989.572,0.262
5,1377.732,2508.899,0.294
6,1176.212,2022.305,0.245
7,1269.630,2216.594,0.263
8,1025.991,1766.750,0.218


'seasonal_median_4w'

,MAE,RMSE,WAPE
fold,,,
1,1133.049,1986.188,0.203
2,973.756,1484.473,0.198
3,689.516,1228.067,0.146
4,544.354,1005.754,0.111
5,592.564,1066.743,0.126
6,634.609,1139.994,0.132
7,541.409,905.971,0.112
8,453.565,732.789,0.096


'General evaluation'

,MAE,RMSE,WAPE
naive,1239.133,2027.330,0.253
seasonal_naive,985.064,1768.086,0.200
seasonal_median_4w,695.353,1193.748,0.141


# Check seasonal naive on closed days

In [8]:
train_dates, val_dates = list(walk_forward_splits(train_part.date))[1]
train = train_part[train_part.date.isin(train_dates)]
val = train_part[train_part.date.isin(val_dates)]
pred = seasonal_naive(train, val)
pred[pred.date.dt.day_name() == "Sunday"].groupby("date").pred.sum()

date
2017-01-08    0.0
2017-01-15    0.0
2017-01-22    0.0
2017-01-29    0.0
Name: pred, dtype: float32

# Conclusions

- Setup: 8 walk-forward folds of 28 days (the last fold ends on 2017-07-18), the 28-day hold-out is not used. Metrics are computed on open days only.
- Seasonal naive (WAPE 0.200) is clearly better than naive (0.253) because the series have a strong weekly cycle.
- Seasonal naive breaks when the last week contains a closed day. In fold 2 the closed Jan 1 2017 is copied to every January Sunday (predicted sales 0), and WAPE rises to 0.473, worse than naive (0.305).
- The seasonal median of the last 4 weeks, with closed days excluded, fixes this: WAPE 0.141 on average, fold 2 drops from 0.473 to 0.198.
- The hardest folds are the holiday ones (folds 1 and 2, December and January, WAPE about 0.20). Baselines know nothing about holidays, so calendar features can win there.
- Baseline to beat for XGBoost and LSTM: MAE 695, RMSE 1194, WAPE 0.141.